# SMS Spam Detection using Linear Regression

**Repositori**: Machine Learning  
**Topik**: Implementasi Linear Regression pada Klasifikasi SMS Spam  
**Dataset**: `SMSSpam.csv`  

---

## Pendahuluan
Proyek ini mendemonstrasikan implementasi **Linear Regression** untuk tugas klasifikasi teks (Spam vs Ham). Meskipun secara teoretis Logistic Regression lebih tepat untuk klasifikasi, notebook ini mengikuti panduan modul praktikum untuk mengeksplorasi penggunaan model regresi linear dalam memprediksi label biner.

### Alur Kerja (Pipeline):
1. **Data Acquisition**: Mengambil data dari CSV.
2. **EDA (Exploratory Data Analysis)**: Menganalisis distribusi dan karakteristik data.
3. **Data Preparation**: Cleaning dan Label Encoding (Ham=0, Spam=1).
4. **Feature Engineering**: Transformasi teks menggunakan TF-IDF.
5. **Modeling**: Training menggunakan Linear Regression.
6. **Evaluation**: Menggunakan Confusion Matrix, Accuracy, Precision, dan Recall.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging
import sys
from pathlib import Path

sys.path.append(str(Path('../../').resolve()))
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

logging.basicConfig(level=logging.INFO, format='%(message)s')
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

DATA_PATH = '../../data/SMSSpam.csv'
CSV_ENCODING = 'latin-1'
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
TEST_SIZE = 0.4
VAL_SIZE_REL = 0.5
TFIDF_MAX_FEATURES = 3000


## 🛠 1. Data Acquisition & Understanding
Memuat dataset dan melihat struktur data awal.

In [ ]:
data_path = Path(DATA_PATH)
if not data_path.exists():
    raise FileNotFoundError(f'Dataset tidak ditemukan di {data_path.resolve()}')
df = pd.read_csv(data_path, names=['Label', 'Message'], encoding=CSV_ENCODING)
logging.info(f'Shape Dataset: {df.shape}')
df.head()


## 📊 2. Exploratory Data Analysis (EDA)
Menganalisis karakteristik dataset sebelum preprocessing.

In [ ]:
logging.info(f'Missing Values:\n{df.isnull().sum()}')

logging.info('Distribusi Label:')
logging.info(f'\n{df["Label"].value_counts()}')
logging.info(f'Persentase:\n{df["Label"].value_counts(normalize=True).mul(100).round(2)}')

df['Message_Length'] = df['Message'].apply(len)
logging.info(f'Statistik Panjang Pesan:\n{df["Message_Length"].describe()}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x='Message_Length', hue='Label', bins=50, kde=True, ax=axes[0])
axes[0].set_title('Distribusi Panjang Pesan')
sns.boxplot(data=df, x='Label', y='Message_Length', palette='viridis', ax=axes[1])
axes[1].set_title('Boxplot Panjang Pesan per Label')
plt.tight_layout()
plt.show()

df.drop(columns=['Message_Length'], inplace=True)


## 🧹 3. Data Preparation & Cleaning
Melakukan encoding pada label dan membersihkan data jika diperlukan.

In [ ]:
df['Label_Num'] = df['Label'].map({'ham': 0, 'spam': 1})

sns.countplot(x='Label', data=df, hue='Label', palette='viridis', legend=False)
plt.title('Distribusi Ham vs Spam')
plt.show()


## 🧪 4. Feature Engineering (TF-IDF)
Mengubah teks pesan menjadi representasi angka menggunakan TF-IDF Vectorizer.

In [ ]:
y = df['Label_Num']

X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    df['Message'], y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp, test_size=VAL_SIZE_REL, random_state=RANDOM_STATE, stratify=y_temp
)

tfidf = TfidfVectorizer(stop_words='english', max_features=TFIDF_MAX_FEATURES)
X_train = tfidf.fit_transform(X_train_text)
X_val = tfidf.transform(X_val_text)
X_test = tfidf.transform(X_test_text)

logging.info(f'Data Training: {X_train.shape[0]} sampel')
logging.info(f'Data Validation: {X_val.shape[0]} sampel')
logging.info(f'Data Testing: {X_test.shape[0]} sampel')


## 🤖 5. Model Training (Linear Regression)
Melatih model dan mencari threshold optimal via ROC Curve pada validation set.

In [ ]:
from utils.evaluation import plot_roc_curve

model = LinearRegression()
model.fit(X_train, y_train)

y_val_continuous = model.predict(X_val)

roc_auc, optimal_threshold, fpr_opt, tpr_opt = plot_roc_curve(
    y_val, y_val_continuous, title='ROC Curve - Validation Set', return_thresholds=True
)

logging.info(f'Threshold Optimal: {optimal_threshold:.3f}')
logging.info(f'TPR (Recall): {tpr_opt:.3f}')
logging.info(f'FPR: {fpr_opt:.3f}')

y_test_continuous = model.predict(X_test)
y_pred = np.where(y_test_continuous >= optimal_threshold, 1, 0)


## 📊 6. Performance Evaluation
Mengevaluasi model menggunakan metrik yang telah didefinisikan di `README.md`.

In [ ]:
from utils.evaluation import plot_confusion_matrix

plot_confusion_matrix(y_test, y_pred, labels=['Ham', 'Spam'], title='Confusion Matrix - Linear Regression (SMS Spam)')

logging.info('--- Laporan Klasifikasi ---')
logging.info(f'\n{classification_report(y_test, y_pred)}')
logging.info(f'Accuracy: {accuracy_score(y_test, y_pred):.2f}')


## 📝 Kesimpulan
Model Linear Regression dengan optimasi threshold via ROC Curve pada validation set mampu memberikan hasil yang cukup baik untuk klasifikasi SMS Spam. Pendekatan ini mengatasi kelemahan threshold tetap 0.5 pada data tidak seimbang (imbalanced). Namun, untuk dataset ini, model berbasis probabilitas seperti Naive Bayes atau Logistic Regression biasanya lebih direkomendasikan.